# FULL-SYN — smoke và benchmark trên máy GPU (F0.7)

[Plan FULL-SYN](../../docs/project/full-syn-train-plan-2026-10-22.md), mục 4 (F0.7) và 4.1. Chạy **một lần trên mỗi máy mới**,
trước notebook `05_full_syn_r<N>`. Kết quả nằm trong `smoke/` của campaign, tách khỏi run thật.

Notebook kiểm:
1. Gói dữ liệu: mọi ảnh mà file nhãn liệt kê đều có; 300 dòng ngẫu nhiên của `SHA256SUMS` vẫn khớp.
2. FSA chạy `smoke_steps` bước, dừng, rồi chạy tiếp từ checkpoint thêm chừng ấy bước (kiểm resume).
3. PDP task 1 chạy `smoke_steps` bước từ detector COCO (như B2).
4. Giây/bước, VRAM tối đa, ước lượng giờ train của một vòng (chưa tính val mỗi epoch).

Xem kết quả rồi mới chạy notebook 05.

In [ ]:
import json, os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(REPO))
from autocheckout import full_syn
from autocheckout.io import load_json
from autocheckout.training_workflow import resolve
SETTINGS_PATH = resolve(os.environ.get('AUTOCHECKOUT_TRAINING_SETTINGS', 'data/training/settings_full_syn_r1.json'))
SETTINGS = load_json(SETTINGS_PATH)
PLAN = full_syn.prepare(SETTINGS)
print('Settings:', SETTINGS_PATH)
print('Campaign:', PLAN['root'], '| round', PLAN['round'], '| dry run' if PLAN['dry_run'] else '| real run')
print('Synthetic config sha256:', PLAN['release_config_sha256'])
print('Train images per task:', PLAN['train_images'])
print(json.dumps({k: PLAN[k] for k in ('resolution', 'batch_size', 'effective_batch', 'num_workers', 'tf32',
                                         'fsa_epochs', 'epochs', 'audit_images')}, indent=1))

## 1. Máy và môi trường

In [ ]:
import platform, subprocess, torch
from autocheckout.runinfo import git_state
git = git_state(REPO)
print('git', git.get('commit'), '| dirty' if git.get('dirty') else '| clean')
print('Python', platform.python_version(), '| torch', torch.__version__, '| CUDA', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout if torch.cuda.is_available() else '')

## 2. Smoke, resume và benchmark

In [ ]:
SMOKE = full_syn.stage_smoke(PLAN, steps=SETTINGS.get('smoke_steps', 200))
print(json.dumps(SMOKE, indent=1))

## 3. Đọc kết quả

- `data.missing_images` phải toàn 0 và `sample_mismatched` = 0.
- `fsa.resume_verification.state_equal` phải là `true`.
- So `training_hours_estimate` với mục 7 của plan (FSA 6–9 giờ, task 1 khoảng 10 giờ, task 2–5 khoảng 33 giờ).
  Lệch nhiều thì cập nhật ước lượng trước khi chạy notebook 05.
- VRAM tối đa phải còn dư so với bộ nhớ GPU.